# Incorporating XML Language Identifiers as Token Metadata

This template duplicates [`load_xml.ipynb`](https://github.com/langeslag/ehtc/blob/main/templates/load_xml.ipynb) but instead of generating a list of tokens, it produces a list of dictionaries encoding a language identifier along with each token form. You can adapt this approach to cover other kinds of metadata, such as part of speech and lemma, to the extent these can be gleaned from the XML. I will provide a separate notebook to demonstrate how metadata may be generated by looping in external taggers.

In [1]:
from pathlib import Path
from lxml import etree
from git import Repo

We'll ascertain the ECHOE repository has been cloned so we have XML documents to work with:

In [2]:
# HTTPS clone point:
remote = 'https://github.com/ECHOEProject/echoe.git'
# Desired target folder name:
local = Path.cwd().parent / 'corpora' / 'echoe'
# Only clone if the target folder doesn't already exist:
if not(local.exists()):
    repo = Repo.clone_from(remote, local)
# Else, just update the working copy from remote:
else:
    repo = Repo(local)
    assert isinstance(repo, Repo)
    repo.remotes.origin.pull()
assert not repo.bare

In [3]:
# Normalization matrix:
substitutions = {
    'ę': 'æ',
    'ƿ': 'w',
    'ẏ': 'y',
    'ſ': 's',
    '': 's', # Using the glyph for descending s, instead of the unicode key point
    'v': 'u',
    'j': 'i',
    '⁊': 'and',
    ' ': '',
    '\n': ''
}

# Token normalization:
def normalize(token):
    # Lowercase:
    token = token.lower()
    for k,v in substitutions.items():
        # Carry out replacements:
        token = token.replace(k, v)
    return token

# Discarding unwanted elements:
def simplify(branch):
    discard = ['abbr', 'am', 'sic', 'del', 'note', 'surplus']  
    # Now we define their text nodes as empty strings:
    query = ['{http://www.tei-c.org/ns/1.0}' + i for i in discard]
    for hit in branch.iter(query):
        hit.text = ''
    return branch

In [4]:
parser = etree.XMLParser(remove_blank_text=True,resolve_entities=True)
corpus_folder = local / 'xml'
corpus = dict()
for file in corpus_folder.glob('*.xml'):
    basename = file.name[:-4]
    tree = etree.parse(file, parser=parser)
    root = simplify(tree.getroot())
    segments = dict()
    for segment in root.iter('{http://www.tei-c.org/ns/1.0}s'):
        # Remember to switch to the default XML namespace to access @xml:id or @xml:lang attributes!
        identifier = segment.get('{http://www.w3.org/XML/1998/namespace}id')
        tokens = []
        for token in segment.iter('{http://www.tei-c.org/ns/1.0}w'):
            # If a word element is marked as the last part of a word, add its text content to the preceding token:
            if token.get('part') == 'F':
                position = len(tokens)-1
                tokens[position]['form'] = tokens[position]['form'] + normalize(etree.tostring(token, method='text', encoding='unicode'))
            else:
                if token.get('{http://www.w3.org/XML/1998/namespace}lang'):
                    token_language = token.get('{http://www.w3.org/XML/1998/namespace}lang')
                else:
                    token_language = token.xpath('ancestor::*[@xml:lang][1]/@xml:lang')[0]
                token_form = normalize(etree.tostring(token, method='text', encoding='unicode'))
                token_dict = {
                    'form': token_form,
                    'lang': token_language
                }
                tokens.append(token_dict)
        segments[identifier] = tokens
    corpus[basename] = segments
        

In [5]:
corpus['309.25']['s309.25.35']

[{'form': 'and', 'lang': 'ang'},
 {'form': 'heo', 'lang': 'ang'},
 {'form': 'þa', 'lang': 'ang'},
 {'form': 'lædde', 'lang': 'ang'},
 {'form': 'on', 'lang': 'ang'},
 {'form': 'sumere', 'lang': 'ang'},
 {'form': 'stowe', 'lang': 'ang'},
 {'form': 'and', 'lang': 'ang'},
 {'form': 'him', 'lang': 'ang'},
 {'form': 'to', 'lang': 'ang'},
 {'form': 'cwæð', 'lang': 'ang'},
 {'form': 'sustinete', 'lang': 'la'},
 {'form': 'hic', 'lang': 'la'},
 {'form': 'et', 'lang': 'la'},
 {'form': 'uigilate', 'lang': 'la'},
 {'form': 'mecum', 'lang': 'la'},
 {'form': 'bidað', 'lang': 'ang'},
 {'form': 'ge', 'lang': 'ang'},
 {'form': 'her', 'lang': 'ang'},
 {'form': 'he', 'lang': 'ang'},
 {'form': 'cwæð', 'lang': 'ang'},
 {'form': 'and', 'lang': 'ang'},
 {'form': 'waciað', 'lang': 'ang'},
 {'form': 'mid', 'lang': 'ang'},
 {'form': 'me', 'lang': 'ang'}]